#### Transform Constructors Data (Bronze to Silver)

**Steps:**
1. Read the raw data from the Bronze table
2. Drop the `url` column
3. Rename `constructorId` to `constructor_id` and `name` to `constructor_name`
4. Remove duplicates based on `constructor_id`
5. Apply title case to `nationality`
6. Save to Silver table

In [0]:
%run ../00-common/01.environment-config 

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

In [0]:
from pyspark.sql import functions

#### Read Bronze Table
Read the raw constructors data from `formula1.bronze.constructors` into a DataFrame.

In [0]:
constructors_df = spark.read.table(bronze_table)
display(constructors_df)


#### Drop Columns
Remove the `url` column - not needed for analysis.

In [0]:
constructors_drop_df = constructors_df.drop('url')

#### Rename Columns
Rename `constructorId` to `constructor_id` and `name` to `constructor_name` using `withColumnsRenamed()`.

In [0]:
constructor_renamed_df = (constructors_drop_df.withColumnsRenamed({'constructorId': 'constructor_id',
 'name': 'constructor_name'}))                                         

#### Remove Duplicates
Drop duplicate rows based on `constructor_id` column.

In [0]:
constructor_distinct_df = constructor_renamed_df.dropDuplicates(["constructor_id"])


In [0]:
display(constructor_distinct_df)

#### Title Case
Convert `nationality` to title case using `initcap()` (e.g., "british" becomes "British").

In [0]:
from pyspark.sql.functions import *
constructor_final_df = (constructor_distinct_df
                        .withColumn('nationality',initcap(col('nationality'))))

#### Write to Silver Table
Save the cleaned DataFrame to `formula1.silver.constructors` in Delta format with overwrite mode.

In [0]:
(
    constructor_final_df
    .write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(silver_table)
)

In [0]:
spark.read.table(silver_table).display()